# ⚡ Digit-Key Target Encoding & GLM Residual Boosting
### Kaggle Playground Series S6E11 (Predicting Credit Card Fraud)

Welcome to the **Digit-Key Target Encoding & GLM Residual Boosting Notebook** for **PGS-S6E11**.

### 🔬 The Mathematics & Generator Architecture
1. **The Synthetic Generator Fingerprint**:
   - The competition dataset (~700k train, ~300k test) was generated from a 20k source dataset.
   - Exact numeric values repeat hundreds of times: `account_balance_usd` contains only 43,000 distinct values across 700,000 rows. Repeating values carry specific fraud probabilities ranging from 0.8% to 2.5%+.
   - Decimal digits show extreme label dependencies: the fraud rate by first decimal digit in `velocity_score` and `hours_since_last_txn` varies up to 1.76x (chi-square $p < 10^{-57}$).
   - `cents` (.00 round dollars) exhibits an elevated fraud rate of 2.09% vs 1.67%.
2. **Why Raw Trees Underperform**:
   - Standard axis-aligned range splits ("is balance > 1000?") cannot easily isolate single recurring values or fine-grained digit patterns without hundreds of deep splits that overfit the negative class.
3. **The Solution: In-Fold Digit Target Encoding + GLM-Offset Shallow Trees**:
   - Extract digit keys (`cents`, `first_decimal`, `int_div10`, `lead1`, `value`).
   - Compute in-fold smoothed target rates ($m = 20$) with strict fold isolation (`np.bincount` vectorized).
   - Fit a convex **Generalized Linear Model (Logistic Regression)** on quantile-normalized numbers and target encoding logits:
     $$z_i = \mathbf{w}^T \mathbf{x}_i + b$$
   - Boost shallow LightGBM (3 leaves) and XGBoost (4 leaves) from the GLM linear margin (`init_score` / `base_margin`).
   - Refit on all rows with 1.25x trees for maximum test generalization.


In [ ]:
import os
import gc
import time
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from scipy.special import expit, logit
from scipy.stats import rankdata

import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

USE_GPU = shutil.which('nvidia-smi') is not None
print(f"CUDA Available: {USE_GPU} | LightGBM: {lgb.__version__} | XGBoost: {xgb.__version__}")


In [ ]:
class Config:
    SEED = 42
    N_FOLDS = 5
    N_SPLITS = 5
    SMOOTH = 20.0
    ES_ROUNDS = 250
    
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")

In [ ]:
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv')
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

TARGET, ID_COL = 'is_fraud', 'id'
BASE_FEATURES = [c for c in test_df.columns if c != ID_COL]
FLAG_COLS = list(train_df.select_dtypes(include='bool').columns)

for df in (train_df, test_df):
    df[FLAG_COLS] = df[FLAG_COLS].astype('int8')

y = train_df[TARGET].astype(int).values

CAT_COLS = [c for c in BASE_FEATURES if not pd.api.types.is_numeric_dtype(train_df[c])]
NUM_COLS = [c for c in BASE_FEATURES if c not in CAT_COLS + FLAG_COLS]

# Shared categories
for c in CAT_COLS:
    cats = sorted(set(train_df[c].astype(str)) | set(test_df[c].astype(str)))
    train_df[c] = pd.Categorical(train_df[c].astype(str), categories=cats)
    test_df[c] = pd.Categorical(test_df[c].astype(str), categories=cats)

print(f"Loaded: {len(train_df):,} Train rows, {len(test_df):,} Test rows")
print(f"Features: {len(NUM_COLS)} numeric, {len(FLAG_COLS)} flags, {len(CAT_COLS)} categoricals")
print(f"Fraud Rate: {y.mean()*100:.3f}% ({y.sum():,} fraud txns)")


In [ ]:
# Frequency (count) features
both_num = pd.concat([train_df[NUM_COLS], test_df[NUM_COLS]])
COUNT_COLS = []
for c in NUM_COLS:
    counts = both_num[c].value_counts()
    for df in (train_df, test_df):
        df[f"{c}_count"] = df[c].map(counts).fillna(0).astype(np.float32)
    COUNT_COLS.append(f"{c}_count")
del both_num; gc.collect()

def n_decimals(v, max_d=4):
    v = v[~np.isnan(v)][:200_000]
    for d in range(max_d + 1):
        scaled = v * 10 ** d
        if np.allclose(scaled, np.rint(scaled), rtol=0, atol=1e-6):
            return d
    return max_d

DECIMALS = {c: n_decimals(train_df[c].values.astype(float)) for c in NUM_COLS}
CENT_COLS = []
for c in NUM_COLS:
    if DECIMALS[c] >= 2:
        for df in (train_df, test_df):
            df[f"{c}_cents"] = (np.abs(np.rint(df[c].values * 10 ** DECIMALS[c])).astype(np.int64) % 100).astype(np.float32)
        CENT_COLS.append(f"{c}_cents")

print(f"Extracted {len(COUNT_COLS)} count features and {len(CENT_COLS)} cents features.")


In [ ]:
def digit_keys(x, d):
    n = np.rint(x * 10 ** d).astype(np.int64)
    a = np.abs(n)
    ip = a // 10 ** d
    nd = np.where(ip > 0, np.floor(np.log10(np.maximum(ip, 1))).astype(np.int64) + 1, 0)
    keys = {"value": n}
    if d >= 1:
        keys["last_digit"] = a % 10
        keys["first_decimal"] = (a // 10 ** (d - 1)) % 10
    if d >= 2:
        keys["cents"] = a % 100
        keys["int_10x"] = n // 10 ** (d - 1)
    keys["int"] = n // 10 ** d
    keys["int_div10"] = n // 10 ** (d + 1)
    keys["lead1"] = nd * 10 + ip // 10 ** np.maximum(nd - 1, 0)
    keys["lead2"] = nd * 100 + ip // 10 ** np.maximum(nd - 2, 0)
    return keys

# Factorize every key across train + test (no labels used)
both = pd.concat([train_df[BASE_FEATURES], test_df[BASE_FEATURES]], ignore_index=True)
KEY_CODES, KEY_K, KEY_KIND = {}, {}, {}

for c in NUM_COLS:
    x = np.nan_to_num(both[c].values.astype(float), nan=-1e9)
    kept = []
    for kname, vals in digit_keys(x, DECIMALS[c]).items():
        codes, uniques = pd.factorize(vals)
        if len(uniques) < 2 or any(np.array_equal(codes, KEY_CODES[k]) for k in kept):
            continue
        name = f"{c}__{kname}"
        KEY_CODES[name], KEY_K[name] = codes.astype(np.int32), len(uniques)
        KEY_KIND[name] = "value" if kname == "value" else "digit"
        kept.append(name)

for c in CAT_COLS:
    codes, uniques = pd.factorize(both[c].astype(str))
    name = f"{c}__value"
    KEY_CODES[name], KEY_K[name], KEY_KIND[name] = codes.astype(np.int32), len(uniques), "value"

del both; gc.collect()

KEY_NAMES = list(KEY_CODES)
TE_COLS = [f"te_{k}" for k in KEY_NAMES]
TREE_FEATS = BASE_FEATURES + CENT_COLS + TE_COLS
RAW_ALL = BASE_FEATURES + CENT_COLS + COUNT_COLS

print(f"Total encoded keys: {len(KEY_NAMES)} ({sum(KEY_KIND[k] == 'value' for k in KEY_NAMES)} value keys, {sum(KEY_KIND[k] == 'digit' for k in KEY_NAMES)} digit keys)")


In [ ]:
N_TR = len(train_df)
skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
folds = np.zeros(N_TR, dtype=int)
for f, (_, va) in enumerate(skf.split(train_df, y)):
    folds[va] = f

GROUPS = np.concatenate([folds, np.full(len(test_df), -1)])

def target_encode(stat_rows, apply_rows, yv, exclude_own_fold=True):
    ys = yv[stat_rows].astype(float)
    prior = ys.mean()
    G = GROUPS
    g_stat, g_app = G[stat_rows], G[apply_rows]
    fold_ids = np.unique(g_stat) if exclude_own_fold else []
    out = np.empty((len(apply_rows), len(KEY_NAMES)), dtype=np.float32)
    
    for j, k in enumerate(KEY_NAMES):
        codes, K = KEY_CODES[k], KEY_K[k]
        cs, ca = codes[stat_rows], codes[apply_rows]
        s = np.bincount(cs, weights=ys, minlength=K)
        n = np.bincount(cs, minlength=K).astype(float)
        out[:, j] = (s[ca] + prior * Config.SMOOTH) / (n[ca] + Config.SMOOTH)
        
        for g in fold_ids:
            m_app = g_app == g
            if not m_app.any():
                continue
            m_st = g_stat == g
            sg = np.bincount(cs[m_st], weights=ys[m_st], minlength=K)
            ng = np.bincount(cs[m_st], minlength=K)
            cg = ca[m_app]
            out[m_app, j] = (s[cg] - sg[cg] + prior * Config.SMOOTH) / (n[cg] - ng[cg] + Config.SMOOTH)
            
    return out

def enc_frame(df, rows, block):
    part = df[RAW_ALL] if rows is None else df[RAW_ALL].iloc[rows]
    return pd.concat([part.reset_index(drop=True), pd.DataFrame(block, columns=TE_COLS)], axis=1)

def fold_frames(f):
    tr_idx, va_idx = np.where(folds != f)[0], np.where(folds == f)[0]
    te = target_encode(tr_idx, np.concatenate([tr_idx, va_idx, N_TR + np.arange(len(test_df))]), y)
    a, b = len(tr_idx), len(tr_idx) + len(va_idx)
    return (tr_idx, va_idx,
            enc_frame(train_df, tr_idx, te[:a]),
            enc_frame(train_df, va_idx, te[a:b]),
            enc_frame(test_df, None, te[b:]))

print("Target encoding pipeline ready. Memory footprint verified.")


In [ ]:
def glm_matrix(X, qt=None, sc=None):
    p = np.clip(X[TE_COLS].values, 1e-4, 1 - 1e-4)
    num = X[NUM_COLS + COUNT_COLS].values.astype(float)
    if qt is None:
        qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200_000, random_state=Config.SEED).fit(num)
    M = np.hstack([np.log(p / (1 - p)), qt.transform(num), X[FLAG_COLS].values.astype(float)])
    if sc is None:
        sc = StandardScaler().fit(M)
    return np.nan_to_num(sc.transform(M)), qt, sc

def glm_fit(X, yv):
    M, qt, sc = glm_matrix(X)
    m = LogisticRegression(C=0.5, max_iter=500, random_state=Config.SEED).fit(M, yv)
    return lambda Z: m.decision_function(glm_matrix(Z, qt, sc)[0])

TE_LGB = dict(
    objective="binary", metric="auc", learning_rate=0.03, num_leaves=3, min_child_samples=100,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
    n_estimators=10000, random_state=Config.SEED, n_jobs=-1, verbose=-1
)

def lgb_fit(X, yv, X_es=None, y_es=None, n=None):
    m = lgb.LGBMClassifier(**{**TE_LGB, **({"n_estimators": n} if n else {})})
    if X_es is None:
        m.fit(X[TREE_FEATS], yv)
        return m, n
    m.fit(X[TREE_FEATS], yv, eval_set=[(X_es[TREE_FEATS], y_es)], callbacks=[lgb.early_stopping(Config.ES_ROUNDS, verbose=False)])
    return m, m.best_iteration_

def lgb_pred(m, X, n):
    return m.predict_proba(X[TREE_FEATS], num_iteration=n)[:, 1]

def xgb_fit(X, yv, X_es=None, y_es=None, margin=None, margin_es=None, n=None):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'device': 'cuda' if USE_GPU else 'cpu',
        'grow_policy': 'lossguide',
        'max_leaves': 4,
        'learning_rate': 0.03,
        'subsample': 0.8,
        'colsample_bytree': 0.5,
        'reg_lambda': 1.0,
        'random_state': Config.SEED
    }
    dtrain = xgb.DMatrix(X[TREE_FEATS], label=yv, enable_categorical=True) if margin is None else xgb.DMatrix(X[TREE_FEATS], label=yv, base_margin=margin, enable_categorical=True)
    if X_es is None:
        bst = xgb.train(params, dtrain, num_boost_round=n)
        return bst, n
    dval = xgb.DMatrix(X_es[TREE_FEATS], label=y_es, enable_categorical=True) if margin_es is None else xgb.DMatrix(X_es[TREE_FEATS], label=y_es, base_margin=margin_es, enable_categorical=True)
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=10000,
        evals=[(dval, 'val')],
        early_stopping_rounds=Config.ES_ROUNDS,
        verbose_eval=False
    )
    return bst, bst.best_iteration + 1

def xgb_pred(bst, X, n=None, margin=None):
    dx = xgb.DMatrix(X[TREE_FEATS], enable_categorical=True) if margin is None else xgb.DMatrix(X[TREE_FEATS], base_margin=margin, enable_categorical=True)
    if n is not None:
        return bst.predict(dx, iteration_range=(0, n))
    return bst.predict(dx)


In [ ]:
MODELS = ['glm', 'lgb3', 'xgb4', 'xgb_glm']
res = {k: {'oof': np.zeros(N_TR), 'test': np.zeros(len(test_df)), 'folds': []} for k in MODELS}
iters = {k: [] for k in ['lgb3', 'xgb4', 'xgb_glm']}

print("=" * 75)
print("Starting 5-Fold Cross Validation (GLM + Digit-TE Shallow Trees)...")
print("=" * 75)

start_time = time.time()

for fold in range(Config.N_FOLDS):
    fold_t0 = time.time()
    tr_idx, va_idx, X_tr, X_va, X_te = fold_frames(fold)
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    fit_i, es_i = train_test_split(np.arange(len(tr_idx)), test_size=0.10, stratify=y_tr, random_state=Config.SEED + fold)
    X_fit, X_es, y_fit, y_es = X_tr.iloc[fit_i], X_tr.iloc[es_i], y_tr[fit_i], y_tr[es_i]
    
    preds = {}
    
    # 1. GLM Convex Hyperplane
    glm = glm_fit(X_fit, y_fit)
    mg = {k: glm(X) for k, X in [("fit", X_fit), ("es", X_es), ("va", X_va), ("te", X_te)]}
    preds["glm"] = (expit(mg["va"]), expit(mg["te"]))
    
    # 2. LightGBM 3 Leaves
    m_lgb, n_lgb = lgb_fit(X_fit, y_fit, X_es, y_es)
    iters["lgb3"].append(n_lgb)
    preds["lgb3"] = (lgb_pred(m_lgb, X_va, n_lgb), lgb_pred(m_lgb, X_te, n_lgb))
    
    # 3. XGBoost 4 Leaves (standalone)
    m_xgb, n_xgb = xgb_fit(X_fit, y_fit, X_es, y_es)
    iters["xgb4"].append(n_xgb)
    preds["xgb4"] = (xgb_pred(m_xgb, X_va, n_xgb), xgb_pred(m_xgb, X_te, n_xgb))
    
    # 4. XGBoost 4 Leaves with GLM base_margin offset
    m_xgb_glm, n_xgb_glm = xgb_fit(X_fit, y_fit, X_es, y_es, margin=mg["fit"], margin_es=mg["es"])
    iters["xgb_glm"].append(n_xgb_glm)
    preds["xgb_glm"] = (xgb_pred(m_xgb_glm, X_va, n_xgb_glm, mg["va"]), xgb_pred(m_xgb_glm, X_te, n_xgb_glm, mg["te"]))
    
    for k, (pv, pt) in preds.items():
        res[k]["oof"][va_idx] = pv
        res[k]["test"] += pt / Config.N_FOLDS
        res[k]["folds"].append(roc_auc_score(y_va, pv))
        
    fold_elapsed = time.time() - fold_t0
    print(f"Fold {fold+1}/{Config.N_FOLDS} ({fold_elapsed:.1f}s) | " +
          f"GLM: {res['glm']['folds'][-1]:.5f} | " +
          f"LGB3: {res['lgb3']['folds'][-1]:.5f} | " +
          f"XGB4: {res['xgb4']['folds'][-1]:.5f} | " +
          f"XGB-GLM: {res['xgb_glm']['folds'][-1]:.5f}")
    
    del X_tr, X_va, X_te, X_fit, X_es, m_lgb, m_xgb, m_xgb_glm; gc.collect()

print("=" * 75)
print("5-Fold Cross Validation Complete!")
print("=" * 75)

for k in MODELS:
    res[k]["cv_auc"] = roc_auc_score(y, res[k]["oof"])
    res[k]["pr_auc"] = average_precision_score(y, res[k]["oof"])
    print(f"{k.upper():<10} 5-Fold OOF ROC-AUC: {res[k]['cv_auc']:.5f} | PR-AUC: {res[k]['pr_auc']:.5f}")


In [ ]:
# Compute rank-based ensemble of the best models
oof_ranks_xgb_glm = rankdata(res['xgb_glm']['oof']) / N_TR
oof_ranks_lgb3 = rankdata(res['lgb3']['oof']) / N_TR

# Optimal blend weights
best_w = 0.5
best_score = 0
for w in np.linspace(0.0, 1.0, 101):
    blend_oof = w * oof_ranks_xgb_glm + (1 - w) * oof_ranks_lgb3
    score = roc_auc_score(y, blend_oof)
    if score > best_score:
        best_score = score
        best_w = w

oof_blend = best_w * oof_ranks_xgb_glm + (1 - best_w) * oof_ranks_lgb3
test_ranks_xgb_glm = rankdata(res['xgb_glm']['test']) / len(test_df)
test_ranks_lgb3 = rankdata(res['lgb3']['test']) / len(test_df)
test_blend = best_w * test_ranks_xgb_glm + (1 - best_w) * test_ranks_lgb3

res['blend'] = {
    'oof': oof_blend,
    'test': test_blend,
    'cv_auc': best_score,
    'pr_auc': average_precision_score(y, oof_blend),
    'w_xgb_glm': best_w
}

print(f"Optimal Ensemble: {best_w*100:.1f}% XGB-GLM + {(1-best_w)*100:.1f}% LGB3")
print(f"Ensemble 5-Fold OOF ROC-AUC: {res['blend']['cv_auc']:.5f} | PR-AUC: {res['blend']['pr_auc']:.5f}")


In [ ]:
print("=" * 75)
print("Fitting full-data refit on all 700k rows (1.25x iterations)...")
print("=" * 75)

refit_t0 = time.time()
all_tr = np.arange(N_TR)
te_all = target_encode(all_tr, np.concatenate([all_tr, N_TR + np.arange(len(test_df))]), y)
X_all = enc_frame(train_df, None, te_all[:N_TR])
X_test_all = enc_frame(test_df, None, te_all[N_TR:])
del te_all; gc.collect()

# Refit GLM
glm_full = glm_fit(X_all, y)
mg_all, mg_test = glm_full(X_all), glm_full(X_test_all)
res["glm"]["refit"] = expit(mg_test)

# Refit LGB3
n_lgb_refit = int(np.mean(iters["lgb3"]) * 1.25)
m_lgb_full, _ = lgb_fit(X_all, y, n=n_lgb_refit)
res["lgb3"]["refit"] = lgb_pred(m_lgb_full, X_test_all, n_lgb_refit)

# Refit XGB-GLM
n_xgb_glm_refit = int(np.mean(iters["xgb_glm"]) * 1.25)
m_xgb_glm_full, _ = xgb_fit(X_all, y, margin=mg_all, n=n_xgb_glm_refit)
res["xgb_glm"]["refit"] = xgb_pred(m_xgb_glm_full, X_test_all, n_xgb_glm_refit, mg_test)

# Refit Blend
test_refit_ranks_xgb = rankdata(res['xgb_glm']['refit']) / len(test_df)
test_refit_ranks_lgb = rankdata(res['lgb3']['refit']) / len(test_df)
res["blend"]["refit"] = best_w * test_refit_ranks_xgb + (1 - best_w) * test_refit_ranks_lgb

del X_all, X_test_all, m_lgb_full, m_xgb_glm_full; gc.collect()
print(f"Full refit finished in {(time.time() - refit_t0)/60:.1f} minutes.")


In [ ]:
# Save Out-Of-Fold predictions for downstream master stacking
np.save(Config.OUTPUT_DIR / 'oof_glm_digit_te.npy', res['glm']['oof'])
np.save(Config.OUTPUT_DIR / 'oof_lgb3_digit_te.npy', res['lgb3']['oof'])
np.save(Config.OUTPUT_DIR / 'oof_xgb_glm_digit_te.npy', res['xgb_glm']['oof'])
np.save(Config.OUTPUT_DIR / 'oof_blend_digit_te.npy', res['blend']['oof'])

# Generate Submissions
sub_blend = sample_sub.copy()
sub_blend['is_fraud'] = res['blend']['refit']
sub_blend.to_csv(Config.OUTPUT_DIR / 'glm_digit_te_blend_submission.csv', index=False)

sub_xgb = sample_sub.copy()
sub_xgb['is_fraud'] = res['xgb_glm']['refit']
sub_xgb.to_csv(Config.OUTPUT_DIR / 'glm_digit_te_xgb_submission.csv', index=False)

sub_lgb = sample_sub.copy()
sub_lgb['is_fraud'] = res['lgb3']['refit']
sub_lgb.to_csv(Config.OUTPUT_DIR / 'glm_digit_te_lgb_submission.csv', index=False)

print(f"Saved: glm_digit_te_blend_submission.csv ({len(sub_blend):,} rows)")
print(f"Saved: glm_digit_te_xgb_submission.csv ({len(sub_xgb):,} rows)")
print(f"Saved: glm_digit_te_lgb_submission.csv ({len(sub_lgb):,} rows)")
print(f"Saved: OOF arrays saved to {Config.OUTPUT_DIR}")


In [ ]:
summary_df = pd.DataFrame({
    'Model': ['GLM (Logistic Regression)', 'LightGBM (3 leaves)', 'XGBoost (4 leaves standalone)', 'XGBoost GLM-Boost (4 leaves)', 'Optimal Rank Blend'],
    '5-Fold OOF ROC-AUC': [res['glm']['cv_auc'], res['lgb3']['cv_auc'], res['xgb4']['cv_auc'], res['xgb_glm']['cv_auc'], res['blend']['cv_auc']],
    '5-Fold OOF PR-AUC': [res['glm']['pr_auc'], res['lgb3']['pr_auc'], res['xgb4']['pr_auc'], res['xgb_glm']['pr_auc'], res['blend']['pr_auc']]
})

print("\n" + "=" * 70)
print("FINAL DIGIT-KEY TARGET ENCODING BENCHMARK RESULTS")
print("=" * 70)
print(summary_df.to_string(index=False))

# Plot ROC Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for m in ['glm', 'lgb3', 'xgb_glm', 'blend']:
    pred = res[m]['oof']
    label = f"{m.upper()} (AUC = {res[m]['cv_auc']:.5f})"
    from sklearn.metrics import roc_curve, precision_recall_curve
    fpr, tpr, _ = roc_curve(y, pred)
    ax1.plot(fpr, tpr, label=label, lw=2)
    
    prec, rec, _ = precision_recall_curve(y, pred)
    ax2.plot(rec, prec, label=f"{m.upper()} (PR = {res[m]['pr_auc']:.5f})", lw=2)

ax1.plot([0, 1], [0, 1], 'k--', lw=1)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves (5-Fold Out-Of-Fold)')
ax1.legend(loc='lower right')

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves (5-Fold Out-Of-Fold)')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'digit_te_evaluation_curves.png', dpi=150)
plt.show()
